# Narrador de Cenas — Colab (GPU) → ONNX → Streamlit
### Pipeline da aula (CNN Residual) adaptado para demo de sala + Telegram

## Fluxo do projeto (3 fases)

1. **Este notebook no Google Colab** — instalar deps, preparar dados, treinar (GPU se disponível), exportar `narrador_cenas.onnx` e **baixar** o arquivo.
2. **PC na aula** — `streamlit run app/streamlit_app.py` + webcam + Telegram (compartilhar a tela).
3. **Depois** — app web pública (upload de vídeo / câmera IP) reusando o mesmo ONNX.

> **Não rode o treino pesado no Python global do Windows** (evita erros como WinError 5 em `cv2.pyd`). Use Colab até o ONNX; no PC só o Streamlit.

## Classes (8)

`opening_door`, `closing_door`, `walking`, `clapping`, `stretching_arm`, `pushing_cart`, `standing_up`, `sitting_down`

Nota: porta vem do **Kinetics-700** (não existe no Kinetics-400). Sentar/levantar = clips próprios ou dataset demo.


## 1. Ambiente Colab

No menu: **Runtime → Change runtime type → GPU** (opcional, acelera o treino).

A célula abaixo usa `%pip` (recomendado no Colab/Jupyter).


In [ ]:
# %pip evita instalar no interpretador errado (melhor que !pip em notebooks)
%pip install -q imagehash onnx onnxscript onnxruntime opencv-python-headless \
  scikit-learn matplotlib pillow tqdm yt-dlp torch torchvision

In [ ]:
import os, sys, random, warnings, json, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('Rodando no Google Colab:', IN_COLAB)

if IN_COLAB:
    # Opcional: gravar no Drive (descomente as 3 linhas abaixo)
    # from google.colab import drive
    # drive.mount('/content/drive')
    # PROJECT = Path('/content/drive/MyDrive/NarradorCenas')
    PROJECT = Path('/content/NarradorCenas')
else:
    PROJECT = Path('.').resolve()

PROJECT.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)
print('PROJECT =', PROJECT.resolve())

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Dados (frames)

No Colab geramos um **dataset demo** das 8 classes (rápido, valida o pipeline).

Para a aula com dados reais, no seu PC:
- `python scripts/download_kinetics_subset.py`
- clips em `data/custom_actions/` (ver `scripts/prepare_custom_actions.md`)
- `python scripts/build_frames_dataset.py`

e depois faça upload da pasta `data/frames` para o Drive/Colab.


In [ ]:
FRAMES_ROOT = PROJECT / 'data' / 'frames'
VIDEO_ROOT = PROJECT / 'data' / 'kinetics_subset'
MODELS_DIR = PROJECT / 'models'
SAMPLE_DIR = PROJECT / 'sample_data'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

CLASSES_SPEC = [
    ('opening_door', (0, 140, 255), 'door_open'),
    ('closing_door', (0, 90, 200), 'door_close'),
    ('walking', (50, 200, 50), 'walk'),
    ('clapping', (220, 80, 80), 'clap'),
    ('stretching_arm', (200, 180, 40), 'stretch'),
    ('pushing_cart', (160, 60, 200), 'push'),
    ('standing_up', (40, 200, 90), 'stand'),
    ('sitting_down', (40, 120, 220), 'sit'),
]

def draw_scene(class_key, t, color):
    img = np.zeros((224, 224, 3), dtype=np.uint8)
    img[:] = (28, 28, 32)
    cv2.rectangle(img, (0, 175), (224, 224), (55, 55, 55), -1)
    if class_key.startswith('door'):
        door_x = 150 + (int(t * 3) if class_key == 'door_open' else 40 - int(t * 3))
        door_x = max(120, min(190, door_x))
        cv2.rectangle(img, (140, 40), (200, 180), (70, 50, 40), -1)
        cv2.rectangle(img, (door_x, 40), (door_x + 45, 180), color, -1)
        cv2.circle(img, (90, 90 + (t % 3)), 14, (230, 200, 180), -1)
    elif class_key == 'walk':
        x = 20 + t * 10
        cv2.circle(img, (x, 90), 14, color, -1)
        cv2.rectangle(img, (x - 10, 104), (x + 10, 160), color, -1)
    elif class_key == 'clap':
        cx, off = 112, 20 - abs(8 - (t % 16))
        cv2.circle(img, (cx, 70), 14, (230, 200, 180), -1)
        cv2.rectangle(img, (cx - 12, 84), (cx + 12, 150), (200, 180, 160), -1)
        cv2.circle(img, (cx - 25 - off, 100), 10, color, -1)
        cv2.circle(img, (cx + 25 + off, 100), 10, color, -1)
    elif class_key == 'stretch':
        cv2.circle(img, (112, 80), 14, (230, 200, 180), -1)
        cv2.rectangle(img, (100, 94), (124, 150), (200, 180, 160), -1)
        cv2.line(img, (124, 110), (180, 90 - t * 3), color, 6)
    elif class_key == 'push':
        x = 30 + t * 8
        cv2.rectangle(img, (x, 120), (x + 50, 160), (100, 100, 100), -1)
        cv2.circle(img, (x - 15, 90), 12, color, -1)
        cv2.rectangle(img, (x - 25, 102), (x - 5, 155), color, -1)
    elif class_key == 'stand':
        cy = 130 - t * 5
        cv2.rectangle(img, (80, 120), (150, 180), (90, 90, 90), -1)
        cv2.circle(img, (115, cy), 16, color, -1)
        cv2.rectangle(img, (100, cy + 16), (130, cy + 65), color, -1)
    else:
        cy = 50 + t * 5
        cv2.rectangle(img, (80, 120), (150, 180), (90, 90, 90), -1)
        cv2.circle(img, (115, cy), 16, color, -1)
        cv2.rectangle(img, (100, cy + 16), (130, cy + 65), color, -1)
    return img

def generate_demo_videos(n_train=16, n_val=4, n_test=4):
    random.seed(SEED)
    for folder, color, key in CLASSES_SPEC:
        counts = {'train': n_train, 'val': n_val, 'test': n_test}
        idx = 0
        for split, n in counts.items():
            out = VIDEO_ROOT / split / folder
            out.mkdir(parents=True, exist_ok=True)
            for _ in range(n):
                path = out / f'demo_{folder}_{idx:03d}.mp4'
                idx += 1
                if path.exists():
                    continue
                writer = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*'mp4v'), 8, (224, 224))
                phase = random.randint(0, 3)
                for t in range(16):
                    writer.write(draw_scene(key, t + phase, color))
                writer.release()
    print('Vídeos demo em', VIDEO_ROOT)

def extract_frames(frames_per_video=6):
    if FRAMES_ROOT.exists():
        shutil.rmtree(FRAMES_ROOT)
    for split in ['train', 'val', 'test']:
        split_dir = VIDEO_ROOT / split
        if not split_dir.exists():
            continue
        for class_dir in sorted(split_dir.iterdir()):
            if not class_dir.is_dir():
                continue
            out_dir = FRAMES_ROOT / split / class_dir.name
            out_dir.mkdir(parents=True, exist_ok=True)
            videos = list(class_dir.glob('*.mp4')) + list(class_dir.glob('*.avi'))
            for video in videos:
                cap = cv2.VideoCapture(str(video))
                total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
                if total <= 0:
                    cap.release(); continue
                idxs = np.linspace(0, total - 1, num=min(frames_per_video, total), dtype=int)
                for i, fi in enumerate(idxs):
                    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
                    ok, frame = cap.read()
                    if ok:
                        cv2.imwrite(str(out_dir / f'{video.stem}_f{i:02d}.jpg'), frame)
                cap.release()
    print('Frames em', FRAMES_ROOT)

# Se já existir data/frames (upload), reutiliza; senão gera demo
if not (FRAMES_ROOT / 'train').exists():
    print('Gerando dataset demo + frames...')
    generate_demo_videos()
    extract_frames()
else:
    print('Usando frames já existentes em', FRAMES_ROOT)

for split in ['train', 'val', 'test']:
    n = len(list((FRAMES_ROOT / split).rglob('*.jpg')))
    print(f'{split}: {n} frames')
print('Classes:', sorted(p.name for p in (FRAMES_ROOT/'train').iterdir() if p.is_dir()))

## 3. Limpeza rápida e amostras


In [ ]:
def verificar_imagens(diretorio):
    rem = []
    for root, _, files in os.walk(diretorio):
        for fname in files:
            if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
            caminho = os.path.join(root, fname)
            try:
                with Image.open(caminho) as img:
                    img.verify()
                with Image.open(caminho) as img:
                    img.load()
            except Exception:
                rem.append(caminho)
                os.remove(caminho)
    return rem

for nome in ['train', 'val', 'test']:
    print(nome, len(verificar_imagens(FRAMES_ROOT/nome)), 'removidas')

classes_dirs = sorted(p.name for p in (FRAMES_ROOT/'train').iterdir() if p.is_dir())
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, cls in zip(axes.ravel(), classes_dirs):
    imgs = list((FRAMES_ROOT/'train'/cls).glob('*.jpg'))
    ax.imshow(Image.open(random.choice(imgs)))
    ax.set_title(cls, fontsize=9)
    ax.axis('off')
plt.suptitle('Amostras — frames de treino', fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Augmentation, DataLoaders e CNN Residual (aula)


In [ ]:
IMG_SIZE, BATCH_SIZE = 128, 32
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Resize((int(IMG_SIZE * 1.15), int(IMG_SIZE * 1.15))),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(12),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

ds_train = ImageFolder(FRAMES_ROOT/'train', transform=transform_train)
ds_val = ImageFolder(FRAMES_ROOT/'val', transform=transform_eval)
ds_test = ImageFolder(FRAMES_ROOT/'test', transform=transform_eval)
loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
loader_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
loader_test = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
classes = ds_train.classes
print('Classes:', classes)
print(f'Treino={len(ds_train)} Val={len(ds_val)} Teste={len(ds_test)}')

In [ ]:
class BlocoResidual(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.bloco = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.projetor = (
            nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm2d(out_ch))
            if stride != 1 or in_ch != out_ch else nn.Identity()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bloco(x) + self.projetor(x))


class CNNResidual(nn.Module):
    def __init__(self, n_classes=8, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1),
        )
        self.features = nn.Sequential(
            BlocoResidual(64, 64),
            BlocoResidual(64, 128, stride=2),
            BlocoResidual(128, 256, stride=2),
        )
        self.classificador = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.classificador(self.features(self.stem(x)))


modelo = CNNResidual(n_classes=len(classes)).to(device)
print('Parâmetros:', sum(p.numel() for p in modelo.parameters() if p.requires_grad))

## 5. Treinamento (AdamW, label smoothing, early stopping)


In [ ]:
LR, NUM_EPOCHS, PACIENCIA = 3e-4, 15, 5
otimizador = torch.optim.AdamW(modelo.parameters(), lr=LR, weight_decay=1e-4)
criterio = nn.CrossEntropyLoss(label_smoothing=0.1)

def treinar_epoca(modelo, loader):
    modelo.train()
    loss_t = ok = n = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        otimizador.zero_grad()
        logits = modelo(x)
        loss = criterio(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        otimizador.step()
        loss_t += loss.item() * x.size(0)
        ok += (logits.argmax(1) == y).sum().item()
        n += x.size(0)
    return loss_t / n, 100 * ok / n

@torch.no_grad()
def avaliar(modelo, loader):
    modelo.eval()
    loss_t = ok = n = 0
    P, Y = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = modelo(x)
        loss = criterio(logits, y)
        pred = logits.argmax(1)
        loss_t += loss.item() * x.size(0)
        ok += (pred == y).sum().item()
        n += x.size(0)
        P.append(pred.cpu()); Y.append(y.cpu())
    return loss_t / n, 100 * ok / n, torch.cat(P).numpy(), torch.cat(Y).numpy()

historico = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
melhor = float('inf')
stale = 0
CKPT = MODELS_DIR / 'melhor_modelo.pth'

for epoca in range(1, NUM_EPOCHS + 1):
    tr_l, tr_a = treinar_epoca(modelo, loader_train)
    va_l, va_a, _, _ = avaliar(modelo, loader_val)
    historico['train_loss'].append(tr_l)
    historico['train_acc'].append(tr_a)
    historico['val_loss'].append(va_l)
    historico['val_acc'].append(va_a)
    mark = ''
    if va_l < melhor:
        melhor, stale = va_l, 0
        torch.save(modelo.state_dict(), CKPT)
        mark = '*'
    else:
        stale += 1
    print(f'Ep {epoca:02d} | T {tr_l:.4f}/{tr_a:.1f}% | V {va_l:.4f}/{va_a:.1f}% {mark}')
    if stale >= PACIENCIA:
        print('Early stopping')
        break

modelo.load_state_dict(torch.load(CKPT, map_location=device, weights_only=True))
print('Checkpoint restaurado.')

In [ ]:
ep = range(1, len(historico['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep, historico['train_loss'], label='Treino')
axes[0].plot(ep, historico['val_loss'], label='Val')
axes[0].legend(); axes[0].set_title('Loss'); axes[0].grid(alpha=0.3)
axes[1].plot(ep, historico['train_acc'], label='Treino')
axes[1].plot(ep, historico['val_acc'], label='Val')
axes[1].legend(); axes[1].set_title('Accuracy'); axes[1].grid(alpha=0.3)
plt.suptitle('Histórico de treinamento', fontweight='bold')
plt.tight_layout(); plt.show()

te_l, te_a, preds, labels = avaliar(modelo, loader_test)
print(f'Teste loss={te_l:.4f} acc={te_a:.2f}%')
print(classification_report(labels, preds, target_names=classes, zero_division=0))

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(confusion_matrix(labels, preds), display_labels=classes).plot(
    ax=ax, cmap='Blues', colorbar=False, xticks_rotation=45
)
plt.tight_layout(); plt.show()

## 6. Narração de um vídeo de exemplo


In [ ]:
NARRATION = {
    'opening_door': 'Na cena, alguém está abrindo a porta.',
    'closing_door': 'Na cena, alguém está fechando a porta.',
    'walking': 'Na cena, uma pessoa está andando.',
    'clapping': 'Na cena, alguém está batendo palmas.',
    'stretching_arm': 'Na cena, alguém está alongando o braço.',
    'pushing_cart': 'Na cena, alguém está empurrando um carrinho.',
    'standing_up': 'Na cena, uma pessoa está se levantando da cadeira.',
    'sitting_down': 'Na cena, uma pessoa está se sentando.',
}

def narrar_video(caminho, modelo, classes, device, every_n=8, max_frames=16):
    cap = cv2.VideoCapture(str(caminho))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    preds = []
    idx = 0
    modelo.eval()
    while len(preds) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        if idx % every_n == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            tensor = transform_eval(Image.fromarray(rgb)).unsqueeze(0).to(device)
            with torch.no_grad():
                probs = F.softmax(modelo(tensor), dim=1).squeeze().cpu()
            i = int(probs.argmax())
            preds.append({
                't': idx / fps,
                'class': classes[i],
                'conf': float(probs[i]),
                'text': NARRATION.get(classes[i], classes[i]),
            })
        idx += 1
    cap.release()
    from collections import Counter
    cnt = Counter(p['class'] for p in preds if p['conf'] >= 0.3)
    if not cnt:
        return 'Cena indefinida.', preds
    dom = cnt.most_common(1)[0][0]
    avg = float(np.mean([p['conf'] for p in preds if p['class'] == dom]))
    return f"{NARRATION.get(dom, dom)} (confiança média: {avg:.0%})", preds

# copia um mp4 por classe para sample_data
for class_dir in sorted((VIDEO_ROOT / 'test').iterdir()):
    if not class_dir.is_dir():
        continue
    vids = list(class_dir.glob('*.mp4'))
    if vids:
        dest = SAMPLE_DIR / f'{class_dir.name}.mp4'
        shutil.copy2(vids[0], dest)

demo = next(SAMPLE_DIR.glob('opening_door.mp4'), None) or next(SAMPLE_DIR.glob('*.mp4'))
resumo, det = narrar_video(demo, modelo, classes, device)
print('Vídeo:', demo.name)
print('Narração:', resumo)
for d in det[:6]:
    print(f"  {d['t']:.1f}s → {d['class']} ({d['conf']:.0%})")

## 7. Exportação ONNX + download (Colab)

Gera `narrador_cenas.onnx` com metadados (classes, img_size, mean/std).  
No Colab, a última parte **baixa o arquivo** para o seu computador.


In [ ]:
import onnx

modelo.eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
onnx_path = MODELS_DIR / 'narrador_cenas.onnx'

torch.onnx.export(
    modelo,
    dummy,
    str(onnx_path),
    input_names=['imagem'],
    output_names=['predicoes'],
    opset_version=18,
    dynamo=False,
)

m = onnx.load(str(onnx_path))
# limpa metadados antigos se reexportar
while len(m.metadata_props):
    del m.metadata_props[0]

meta = {
    'task': 'action_recognition_frame',
    'classes': json.dumps(classes),
    'img_size': str(IMG_SIZE),
    'mean': json.dumps(MEAN),
    'std': json.dumps(STD),
    'color_mode': 'RGB',
    'framework': 'PyTorch',
    'architecture': 'CNNResidual',
    'dataset': 'kinetics_subset_classroom',
}
for k, v in meta.items():
    p = m.metadata_props.add()
    p.key, p.value = k, v
onnx.save(m, str(onnx_path))

print(f'ONNX: {onnx_path}')
print(f'Tamanho: {onnx_path.stat().st_size / 1e6:.2f} MB')
print('Classes:', classes)

# Download no Colab
if IN_COLAB:
    from google.colab import files
    files.download(str(onnx_path))
    print('Download iniciado: narrador_cenas.onnx')
else:
    print('Fora do Colab: copie o arquivo de', onnx_path)

## 8. Próximos passos (fora do Colab)

### Na aula (PC + tela compartilhada)

1. Coloque `narrador_cenas.onnx` em:
   `Entrega 3 - Narrador de Cenas/models/`
2. No terminal (com `.venv`):

```bash
cd "Entregas - Humberto Nogueira do Carmo/Entrega 3 - Narrador de Cenas"
python -m venv .venv
# Windows: .venv\Scripts\activate
source .venv/bin/activate
pip install -r requirements.txt
streamlit run app/streamlit_app.py
```

3. Modos: **Arquivo** (exemplos) ou **Webcam** (porta / andar / levantar / palmas).
4. Telegram: ative monitoramento e informe token + chat id.

### App web pública (fase futura)

Reutilize o mesmo ONNX em um serviço (Streamlit Cloud / FastAPI) com:
- upload de vídeo
- URL de câmera IP / RTSP → frames → narração

### Windows — se der WinError 5 em `cv2.pyd`

Não treine no Python global. Use Colab para o ONNX; no PC só o venv do Streamlit. Se precisar reinstalar OpenCV: feche o Cursor/kernel, `pip uninstall opencv-python opencv-python-headless` e reinstale no `.venv`.
